# nyaya — hydrate Supabase

End-to-end hydration of the remote Supabase corpus for the nyaya MCP server:
Constitution, bare acts (HuggingFace), Sanhitas (PRS PDFs), landmark judgments,
cross-references, and **pgvector embeddings** (BAAI/bge-large-en-v1.5, 1024-d).

**Run from the `mcp/` directory** so relative paths (`data/manual/...`, `scripts/schema.sql`)
resolve the same way the `nyaya-ingest` CLI does.

The notebook reuses the existing `nyaya.scripts.ingest_*` functions as a library —
no ingestion logic is duplicated. The only new code is the enriched-text embedding
cell (sections/articles/judgments get a `act | ref | title` prefix for better retrieval).

**Execution provider**: CUDAExecutionProvider (NVIDIA GPU) with automatic CPU fallback.
fastembed passes the providers list to onnxruntime, which picks the first usable one.

> Idempotent: every ingest step upserts on conflict, so re-running the notebook
> refreshes the corpus without manual cleanup. The schema cell is also idempotent.

## 1. Setup & environment

In [ ]:
import os, sys, platform, subprocess, pathlib

assert pathlib.Path('scripts/schema.sql').is_file(), (
    'Run this notebook from the mcp/ directory (so scripts/schema.sql resolves).'
)

from nyaya.config import get_settings
from nyaya.scripts.db import IngestDB

settings = get_settings()
print('Python      :', sys.version.split()[0])
print('Platform    :', platform.platform())
print('DATABASE_URL :', settings.database_url[:45] + '...')
print('CWD         :', os.getcwd())

In [ ]:
# GPU + ONNX Runtime providers (decides CPU vs CUDA for the embedding step)
try:
    import onnxruntime as ort
    print('onnxruntime  :', ort.__version__)
    print('providers    :', ort.get_available_providers())
except ImportError:
    print('onnxruntime  : not installed')

try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap',
         '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    print('GPU          :', r.stdout.strip() if r.returncode == 0 else 'n/a')
except FileNotFoundError:
    print('GPU          : n/a (nvidia-smi not found)')

In [ ]:
# Open a single IngestDB connection reused across all ingest cells.
db = IngestDB()
db.connect()
print('Connected to Supabase.')

## 2. Apply schema (idempotent)

Creates tables, indexes, the `documents` view, and the `vector` extension. Safe to
run multiple times. Embedding columns are `vector(1024)` for bge-large-en-v1.5.

In [ ]:
db.apply_schema()  # reads scripts/schema.sql
db.commit()
print('Schema applied.')

# Confirm embedding dimension is 1024.
rows = db.fetch_all("""
    select attrelid::regclass as tbl, atttypmod
    from pg_attribute where attname='embedding' and atttypmod > 0
    order by tbl
""")
for r in rows:
    print(f'  {r["tbl"]}: vector({r["atttypmod"]})')

## 3. Ingest Constitution

Articles 1–395 + Preamble from the `indianconstitution` PyPI package (Apache-2.0),
plus schedules and amendments from `data/manual/`.

In [ ]:
from nyaya.scripts.ingest_constitution import ingest_constitution
ingest_constitution(db)

## 4. Ingest bare acts (HuggingFace)

IPC, CrPC, CPC, Evidence Act, and commercial statutes (Companies, IGST, CGST, ITAct,
Arbitration, ConsumerProtection) from `mratanusarkar/Indian-Laws`. First run downloads
the dataset; subsequent runs use the HF cache.

In [ ]:
from nyaya.scripts.ingest_bare_acts import ingest_bare_acts
ingest_bare_acts(db)

## 5. Ingest IPC / Evidence Act / CPC (civictech JSON)

The HuggingFace `mratanusarkar/Indian-Laws` dataset is sparse for these three acts
(IPC has only 12 of ~511 sections). This cell loads the full verbatim section text
from the `civictech-India/Indian-Law-Penal-Code-Json` GitHub repo, overwriting the
HF stubs. Must run AFTER the bare-acts cell so the full text wins the upsert.


In [ ]:
from nyaya.scripts.ingest_civictech import ingest_civictech
ingest_civictech(db)


## 6. Ingest Sanhitas (2023)

BNS, BNSS, BSA from PRS PDFs (CC BY 4.0). Downloads each PDF, extracts text with pypdf,
and splits into sections using heading-based heuristics.

In [ ]:
from nyaya.scripts.ingest_sanhitas import ingest_sanhitas
ingest_sanhitas(db)

## 7. Ingest landmark judgments

Curated YAML in `data/manual/judgments.yaml`. Currently 5 cases (Kesavananda, Maneka,
Puttaswamy, Shah Bano, Navtej Singh Johar) with summary + placeholder body text.

In [ ]:
from nyaya.scripts.ingest_judgments import ingest_judgments
ingest_judgments(db)

## 8. Build cross-references

Two sources: the manual IPC↔BNS map (`data/manual/ipc_bns_map.yaml`) and a regex scan
of all section text for phrases like `section 65 of the Indian Evidence Act` and
`Article 21 of the Constitution`.

In [ ]:
from nyaya.scripts.build_cross_refs import build_cross_refs
build_cross_refs(db)

## 9. Corpus counts so far

In [ ]:
db.print_counts()

## 10. Build embeddings (bge-large-en-v1.5, 1024-d, CUDA + CPU fallback)

The core of the hydration. For each document we embed an **enriched** string that
prefixes the raw text with act / reference / title context — this gives the model
more signal than the bare text column and improves retrieval.

- Sections: `"Act: {act} | s. {number} | {title}\n{text}"`
- Articles: `"art. {number} | {title}\n{text}"`
- Judgments: `"{case_name} ({citation})\n{summary}\n{text}"`

Each text is truncated to 8000 chars (matches the existing CLI behaviour). The query
path (`nyaya.embeddings.embed_query`) is unchanged — it embeds the raw query string,
which the model handles natively.

Provider order: `['CUDAExecutionProvider', 'CPUExecutionProvider']`. ORT picks the
first usable one — CUDA on NVIDIA GPUs (RTX 5060 Ti), CPU elsewhere. If Blackwell
(compute 12.0) isn't yet supported by the installed onnxruntime, ORT silently falls
back to CPU; the warning below shows which provider won.

In [ ]:
from fastembed import TextEmbedding

MODEL = 'BAAI/bge-large-en-v1.5'
PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
model = TextEmbedding(model_name=MODEL, providers=PROVIDERS)
print(f'Loaded {MODEL}.')
# Surface which provider ORT actually selected (CUDA vs CPU fallback).
try:
    sess = model._model._session  # fastembed internal: onnxruntime InferenceSession
    print('Active providers:', sess.get_providers())
except Exception:
    print('(could not introspect active provider — CUDA attempted, CPU fallback automatic)')

In [ ]:
# Enriched-text builders + fetch queries for each table.
MAX_CHARS = 8000

def enrich_section(act, number, title, text):
    head = f'Act: {act} | s. {number}'
    if title:
        head += f' | {title}'
    return (head + '\n' + (text or ''))[:MAX_CHARS]

def enrich_article(number, title, text):
    head = f'art. {number}'
    if title:
        head += f' | {title}'
    return (head + '\n' + (text or ''))[:MAX_CHARS]

def enrich_judgment(case_name, citation, summary, text):
    head = case_name
    if citation:
        head += f' ({citation})'
    parts = [head]
    if summary:
        parts.append(summary)
    parts.append(text or '')
    return '\n'.join(parts)[:MAX_CHARS]

# Fetch rows for each table. Each returns list[dict] with 'id' + the fields the
# enricher needs. Order matters only for progress reporting.
section_rows = db.fetch_all("""
    select s.id, a.short_name as act, s.number, s.title, s.text
    from sections s join acts a on a.id = s.act_id
""")
article_rows = db.fetch_all("select id, number, title, text from articles")
judgment_rows = db.fetch_all("select id, case_name, citation, summary, text from judgments")

print(f'Documents to embed: sections={len(section_rows)}, '
      f'articles={len(article_rows)}, judgments={len(judgment_rows)}')

In [ ]:
# Embed + upsert, robust against Supabase idle-connection drops.
# Pattern: embed ALL texts in memory first (no DB connection held during the slow
# CPU embedding loop), then upsert vectors in quick batches with a FRESH
# connection per batch. Holding one connection open across ~3900 CPU embeddings
# takes ~40 min and Supabase's pooler drops idle connections mid-run.
from tqdm import tqdm
import psycopg
from nyaya.config import get_settings

_URL = get_settings().database_url
BATCH = 200

def embed_and_store(rows, enricher, table, id_field, batch_size=64):
    if not rows:
        print(f'  {table}: no rows, skipping.')
        return 0
    texts = [enricher(**{k: r[k] for k in r if k != 'id'}) for r in rows]
    # Phase A: embed all texts in memory (no DB connection held here).
    print(f'  {table}: embedding {len(texts)} texts...')
    vectors = list(model.embed(texts, batch_size=batch_size))
    # Phase B: upsert in batches, fresh connection per batch.
    total = 0
    for i in range(0, len(rows), BATCH):
        chunk = rows[i:i+BATCH]
        vecs = vectors[i:i+BATCH]
        conn = psycopg.connect(_URL, connect_timeout=15)
        cur = conn.cursor()
        for r, v in zip(chunk, vecs):
            cur.execute(
                f'''insert into {table}_embeddings ({table}_id, embedding)
                   values (%s, %s::vector)
                   on conflict ({table}_id) do update set embedding = excluded.embedding''',
                (str(r[id_field]), v.tolist()),
            )
        conn.commit(); conn.close()
        total += len(chunk)
        print(f'  {table}: {total}/{len(rows)} upserted.')
    return total

n_sec = embed_and_store(section_rows, enrich_section, 'section', 'id')
n_art = embed_and_store(article_rows, enrich_article, 'article', 'id')
n_jud = embed_and_store(judgment_rows, enrich_judgment, 'judgment', 'id')
print(f'\nEmbedded: sections={n_sec}, articles={n_art}, judgments={n_jud}')


## 11. Sanity checks against the live DB

In [ ]:
# Final counts — embeddings should match their parent tables.
final = db.fetch_all("""
    select 'acts' as k, count(*) as n from acts
    union all select 'sections', count(*) from sections
    union all select 'articles', count(*) from articles
    union all select 'judgments', count(*) from judgments
    union all select 'amendments', count(*) from amendments
    union all select 'schedules', count(*) from schedules
    union all select 'cross_refs', count(*) from cross_refs
    union all select 'section_embeddings', count(*) from section_embeddings
    union all select 'article_embeddings', count(*) from article_embeddings
    union all select 'judgment_embeddings', count(*) from judgment_embeddings
""")
counts = {r['k']: int(r['n']) for r in final}
import json; print(json.dumps(counts, indent=2))
assert counts['section_embeddings'] == counts['sections'], 'section embeddings mismatch'
assert counts['article_embeddings'] == counts['articles'], 'article embeddings mismatch'
assert counts['judgment_embeddings'] == counts['judgments'], 'judgment embeddings mismatch'
print('Embeddings match parent tables.')

# Dimension check — vectors must be 1024.
dim = db.fetch_all('select vector_dims(embedding) as d from section_embeddings limit 1')[0]['d']
print(f'Embedding dimension: {dim}')
assert dim == 1024, f'expected 1024, got {dim}'

In [ ]:
# Semantic search sample — should retrieve Puttaswamy + Article 21 for 'right to privacy'.
from nyaya.embeddings import embed_query
from nyaya.db import semantic_search_all

q = 'right to privacy'
emb = embed_query(q)
hits = semantic_search_all(emb, limit=5)
print(f'Semantic query: {q!r} -> {len(hits)} hits')
for h in hits:
    print(f'  {h.act:14s} {h.ref:14s} rank={h.rank:.3f}  {h.title or ""}')

In [ ]:
# Full-text search sample — should retrieve IPC s.302 / BNS s.103 for 'murder'.
from nyaya.db import search_all

q = 'murder'
hits = search_all(q, limit=3)
print(f'FTS query: {q!r} -> {len(hits)} hits')
for h in hits:
    print(f'  {h.act:14s} {h.ref:14s} rank={h.rank:.3f}  {h.title or ""}')
    print(f'     ... {h.snippet[:120]}')

In [ ]:
db.close()
print('Hydration complete. DB connection closed.')